Encode ratings into sentiment categories (positive, neutral, negative) by defining dynamic rating thresholds based on statistical distribution, validating encoding using sentiment analysis, and testing multiple encoding schemes for best predictive power.

Analyze rating distribution

In [1]:
import pandas as pd
import numpy as np

# Load Task 19 final dataset
df = pd.read_csv("task19_final_dataset.csv")

print("Dataset shape:", df.shape)
print("\nMissing ratings:", df["rating"].isna().sum())

# Rating frequency and percentage
rating_distribution = pd.DataFrame({
    "Count": df["rating"].value_counts().sort_index(),
    "Percentage": (
        df["rating"].value_counts(normalize=True).sort_index() * 100
    )
})

display(rating_distribution.round(2))

# Statistical summary
print("\nRating statistics:")
display(df["rating"].describe().to_frame())

print("Skewness:", round(df["rating"].skew(), 3))
print("Kurtosis:", round(df["rating"].kurtosis(), 3))

Dataset shape: (23483, 23)

Missing ratings: 0


,Count,Percentage
rating,,
1,842,3.59
2,1565,6.66
3,2871,12.23
4,5077,21.62
5,13128,55.90



Rating statistics:


,rating
count,23483.000000
mean,4.195929
std,1.110064
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


Skewness: -1.313
Kurtosis: 0.804


Create sentiment categories using multiple schemes

In [3]:
# Work on a copy
task22_df = df.copy()

# Remove rows without a rating
task22_df = task22_df.dropna(subset=["rating"]).copy()

# Ensure rating is numeric
task22_df["rating"] = pd.to_numeric(
    task22_df["rating"], errors="coerce"
)
task22_df = task22_df.dropna(subset=["rating"]).copy()

rating = task22_df["rating"]

# Scheme 1: Standard rating thresholds
task22_df["sentiment_standard"] = np.select(
    [
        rating <= 2,
        rating == 3,
        rating >= 4
    ],
    [
        "Negative",
        "Neutral",
        "Positive"
    ],
    default="Neutral"
)

# Scheme 2: Quantile-based thresholds
q25 = rating.quantile(0.25)
q75 = rating.quantile(0.75)

task22_df["sentiment_quantile"] = np.select(
    [
        rating <= q25,
        rating >= q75
    ],
    [
        "Negative",
        "Positive"
    ],
    default="Neutral"
)

# Scheme 3: Dynamic mean and standard deviation thresholds
mean_rating = rating.mean()
std_rating = rating.std()

lower_threshold = mean_rating - (std_rating / 2)
upper_threshold = mean_rating + (std_rating / 2)

task22_df["sentiment_mean_based"] = np.select(
    [
        rating < lower_threshold,
        rating > upper_threshold
    ],
    [
        "Negative",
        "Positive"
    ],
    default="Neutral"
)

# Display thresholds
print("Rating mean:", round(mean_rating, 3))
print("Rating standard deviation:", round(std_rating, 3))
print("Mean-based lower threshold:", round(lower_threshold, 3))
print("Mean-based upper threshold:", round(upper_threshold, 3))
print("Quantile thresholds:", q25, q75)

# Compare sentiment distributions
for col in [
    "sentiment_standard",
    "sentiment_quantile",
    "sentiment_mean_based"
]:
    print(f"\n{col} distribution:")
    display(
        task22_df[col]
        .value_counts()
        .rename("Count")
        .to_frame()
        .assign(
            Percentage=lambda x:
            (x["Count"] / len(task22_df) * 100).round(2)
        )
    )


Rating mean: 4.196
Rating standard deviation: 1.11
Mean-based lower threshold: 3.641
Mean-based upper threshold: 4.751
Quantile thresholds: 4.0 5.0

sentiment_standard distribution:


,Count,Percentage
sentiment_standard,,
Positive,18205,77.52
Neutral,2871,12.23
Negative,2407,10.25



sentiment_quantile distribution:


,Count,Percentage
sentiment_quantile,,
Positive,13128,55.9
Negative,10355,44.1



sentiment_mean_based distribution:


,Count,Percentage
sentiment_mean_based,,
Positive,13128,55.90
Negative,5278,22.48
Neutral,5077,21.62


Compare category distributions

In [4]:
sentiment_columns = [
    'sentiment_standard',
    "sentiment_quantile",
    "sentiment_mean_based"
]

for col in sentiment_columns:
    print(f"\n{col} distribution:")
    display(task22_df[col].value_counts().rename("Count").to_frame()
        .assign(
            Percentage=lambda x:
            (x["Count"] / len(task22_df) * 100).round(2)
        )
    )


sentiment_standard distribution:


,Count,Percentage
sentiment_standard,,
Positive,18205,77.52
Neutral,2871,12.23
Negative,2407,10.25



sentiment_quantile distribution:


,Count,Percentage
sentiment_quantile,,
Positive,13128,55.9
Negative,10355,44.1



sentiment_mean_based distribution:


,Count,Percentage
sentiment_mean_based,,
Positive,13128,55.90
Negative,5278,22.48
Neutral,5077,21.62


Validate ratings against review sentiment

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [6]:
# Map VADER's sentiment labels to the same category names
vader_map = {
    "negative": "Negative",
    "neutral": "Neutral",
    "positive": "Positive"
}

task22_df['vader_label_normalized'] = (
    task22_df["vader_sentiment_label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(vader_map)
)

validation_df = task22_df.dropna(subset=["vader_label_normalized"]).copy()

for col in sentiment_columns:
    print(f"\nValidation: {col}")

    print("Agreement with VADER: ", round(
        accuracy_score(
            validation_df["vader_label_normalized"],
            validation_df[col]
        ),
        4
    ))

    print(classification_report(
        validation_df['vader_label_normalized'],
        validation_df[col],
        labels=["Negative", "Neutral", "Positive"],
        zero_division=0
    ))


Validation: sentiment_standard
Agreement with VADER:  0.7562
              precision    recall  f1-score   support

    Negative       0.26      0.47      0.33      1334
     Neutral       0.05      0.12      0.07      1108
    Positive       0.93      0.81      0.87     21041

    accuracy                           0.76     23483
   macro avg       0.41      0.46      0.42     23483
weighted avg       0.85      0.76      0.80     23483


Validation: sentiment_quantile
Agreement with VADER:  0.5767
              precision    recall  f1-score   support

    Negative       0.11      0.89      0.20      1334
     Neutral       0.00      0.00      0.00      1108
    Positive       0.94      0.59      0.72     21041

    accuracy                           0.58     23483
   macro avg       0.35      0.49      0.31     23483
weighted avg       0.85      0.58      0.66     23483


Validation: sentiment_mean_based
Agreement with VADER:  0.578
              precision    recall  f1-score   suppo

Test predictive power

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [9]:

# Use the original review text as model input
task22_df["model_text"] = (
    task22_df["review_title"].fillna("").astype(str)
    + " "
    + task22_df["review_text"].fillna("").astype(str)
).str.strip()

# Exclude empty reviews
model_df = task22_df[
    task22_df["model_text"].str.len() > 0
].copy()

results = []

for target_col in sentiment_columns:
    x = model_df["model_text"]
    y = model_df[target_col]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42, stratify=y)

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=2,
            stop_words="english"
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    model.fit(x_train, y_train)
    predictions = model.predict(x_test)

    results.append({
        "Encoding Scheme": target_col,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro"),
        "Train Rows": len(x_train),
        "Test Rows": len(x_test)
    })

results_df = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)

display(results_df.round(4))

,Encoding Scheme,Accuracy,Macro F1,Train Rows,Test Rows
1,sentiment_quantile,0.8037,0.8003,18786,4697
2,sentiment_mean_based,0.6994,0.6583,18786,4697
0,sentiment_standard,0.8039,0.6316,18786,4697


Save the Task 22 outputs

In [11]:
# Save Task 22 final dataset
task22_df.to_csv("task22_final_dataset.csv",index=False)

# Save predictive performance comparison
results_df.to_csv("task22_encoding_comparison.csv",index=False)

print("Task 22 completed and files saved successfully!")
print("Final dataset shape:", task22_df.shape)
print("Final dataset: task22_final_dataset.csv")
print("Comparison file: task22_encoding_comparison.csv")

Task 22 completed and files saved successfully!
Final dataset shape: (23483, 28)
Final dataset: task22_final_dataset.csv
Comparison file: task22_encoding_comparison.csv


## Conclusion – Task 22: Sentiment Category Encoding

In this task, sentiment categories were generated from the `rating` column of the Women’s Clothing Reviews dataset containing 23,483 records and 23 columns, with no missing ratings. The rating distribution was negatively skewed (skewness = -1.313), indicating that customers generally gave high ratings. In particular, 55.90% of reviews received a rating of 5, while only 3.59% received a rating of 1.

Three sentiment classification schemes were implemented and compared. The standard method classified 77.52% of reviews as Positive, 12.23% as Neutral, and 10.25% as Negative. The quantile-based method produced 55.90% Positive and 44.10% Negative reviews, with no Neutral category because the 75th percentile was equal to the maximum rating of 5. The mean-based method classified 55.90% as Positive, 21.62% as Neutral, and 22.48% as Negative.

Validation against VADER sentiment labels showed that the standard scheme achieved the highest agreement at 75.62%, compared with 57.67% for the quantile-based method and 57.80% for the mean-based method. However, the standard scheme's macro F1-score against VADER was 0.42, demonstrating weaker performance on the minority sentiment classes than on the Positive class.

In the text-based Logistic Regression evaluation using TF-IDF features, the quantile-based scheme achieved an accuracy of 0.8037 and the highest Macro F1-score of 0.8003. The standard scheme achieved a slightly higher accuracy of 0.8039 but a lower Macro F1-score of 0.6316. The mean-based scheme achieved an accuracy of 0.6994 and a Macro F1-score of 0.6583.

Overall, the standard scheme showed the strongest agreement with VADER, while the quantile-based scheme achieved the highest predictive Macro F1-score. However, the quantile-based scheme failed to generate a Neutral class because of the dataset's rating distribution. Therefore, the standard scheme is a more interpretable choice for preserving all three sentiment categories, while the quantile-based scheme should not be selected as a three-class encoding method in its current form. The final dataset was saved as `task22_final_dataset.csv`, containing 23,483 rows and 28 columns.